# Explore Gherkin AST (Sandbox)

This notebook uses the **gherkin-official** library to parse a `.feature` file
and inspect the Python dictionary (the Gherkin AST) that the parser produces.

**Goal:** Understand the nested structure — `Feature → Children/Scenarios → Steps` —
before writing the deterministic mapping logic for `src/ast_gen_B.py`.

In [ ]:
import json
from pathlib import Path

from gherkin.parser import Parser
from gherkin.token_scanner import TokenScanner

In [ ]:
# Pick a .feature file to explore
FEATURE_FILE = Path("data/gherkin/signal_light_demo_api.feature")
assert FEATURE_FILE.is_file(), f"Feature file not found: {FEATURE_FILE.resolve()}"

feature_text = FEATURE_FILE.read_text(encoding="utf-8")
print(f"=== {FEATURE_FILE.name} ===\n")
print(feature_text)

In [ ]:
# Parse with gherkin-official
parser = Parser()
scanner = TokenScanner(feature_text)
gherkin_ast = parser.parse(scanner)

print(json.dumps(gherkin_ast, indent=2))

## Structure Observations

Key fields in the parsed dictionary:

- `feature.name` — Feature title from the `Feature:` line
- `feature.description` — The paragraph after the title
- `feature.children` — Array of child nodes; each scenario lives in
  `child['scenario']`
- `scenario.name` — Scenario name (e.g. `"Operator starts the system"`)
- `scenario.steps` — Array of step dicts; each has `keyword` (e.g.
  `"When "`, `"Then "`) and `text` (the step body)

In [ ]:
# Quick structural summary — convenience for exploration
feature = gherkin_ast["feature"]
print(f"Feature title : {feature['name']}")
print(f"Description   : {feature['description'].strip()}")
print(f"# Scenarios   : {len(feature['children'])}\n")

for child in feature["children"]:
    scenario = child["scenario"]
    print(f"  Scenario: {scenario['name']}")
    for step in scenario["steps"]:
        kw = step["keyword"].strip()
        txt = step["text"]
        print(f"    {kw}: {txt}")
    print()

In [ ]:
# Try the sample_control file as well — bigger example
FEATURE2 = Path("data/gherkin/sample_control_api.feature")
if FEATURE2.is_file():
    text2 = FEATURE2.read_text(encoding="utf-8")
    ast2 = Parser().parse(TokenScanner(text2))
    print(json.dumps(ast2, indent=2))